# 시편 PatchCore 3회 교차 검증

**런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택한 뒤 위에서부터 순서대로 실행하세요.

구글 드라이브 구조:
```
내 드라이브/specimen_kit/
  patchcore_cv.py  specimens.csv  holes.csv  masks/(90개)
  raw/  ← HEIC 45장
```

In [ ]:
# 1. 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/specimen_kit'

In [ ]:
# 2. GPU 확인 + 라이브러리 설치
!nvidia-smi -L
!pip -q install pillow-heif

In [ ]:
# 3. 파일 점검 (없는 파일이 있으면 여기서 멈춤)
import os, csv
rows = list(csv.DictReader(open(f'{BASE}/specimens.csv', encoding='utf-8')))
missing = [r['file'] for r in rows if not os.path.exists(f"{BASE}/raw/{r['file']}")]
print('시편', len(rows), '장 / 없는 파일:', missing if missing else '없음')
print('마스크', len(os.listdir(f'{BASE}/masks')), '개 (90개여야 함)')
assert not missing

In [ ]:
# 4. 실행 (T4 기준 수 분)
!cd "$BASE" && python patchcore_cv.py --kit . --raw raw --out results 2>&1 | grep -v -i warning

In [ ]:
# 5. 결과 요약표
import json
S = json.load(open(f'{BASE}/results/summary.json'))
A = S['aggregate']
def pct(x): return f"{x[0]}/{x[1]} ({100*x[0]/x[1]:.0f}%)" if x[1] else '-'
for tn, name in (('strict','엄격 기준(학습 정상 최대값)'), ('p999','완화 기준(학습 정상 99.9%)')):
    a = A[tn]
    print(f'\n[{name}]')
    print('  정상 A 과검(장):', pct(a['false_alarm_A']), '| 정상 B 과검(장):', pct(a['false_alarm_B']))
    print('  핀홀 시편 구멍 밖 과검(장):', pct(a['pinhole_background_alarm_imgs']))
    print('  구멍 검출(1mm 이상):', pct(a['holes_detected']))
    for k, v in a['holes_by_size'].items(): print(f'    {k} mm: {pct(v)}')
    print('  1mm 미만·안 보임 구멍 반응:', pct(a['small_holes_detected']))
    print('  이물·줄무늬 시편 반응(3회 중):', a['defect_imgs_over_thr'])
print('\n이미지 단위 AUROC(정상 vs 결함) 평균:', round(A['image_auroc_mean'], 3))
for f, v in S['folds'].items(): print(f, '엄격 기준값', round(v['strict']['threshold'], 3), '← 최대 학습 시편:', max(v['loo_max'], key=v['loo_max'].get))

In [ ]:
# 6. 히트맵 몇 장 보기 (빨강=이상 점수 높음, 하늘색 테두리=엄격 기준 초과, 흰 원=정답 구멍)
from IPython.display import Image as IMG, display
import glob
for p in sorted(glob.glob(f'{BASE}/results/heatmaps/A_fold1_P*.jpg'))[:4] + sorted(glob.glob(f'{BASE}/results/heatmaps/A_fold1_N*.jpg'))[:4]:
    print(os.path.basename(p)); display(IMG(p, width=260))

## 결과를 Claude에게 보낼 때
`results` 폴더를 통째로 압축해서 올려주세요. 압축이 어려우면 **`results/summary.json`** 과 **`results/A_fold1/images.csv`, `holes.csv`** (fold2·3도) 만 올려도 됩니다.

In [ ]:
# 7. results 폴더 압축 (드라이브의 specimen_kit/results.zip 으로 저장)
!cd "$BASE" && zip -qr results.zip results -x '*.npy' && ls -la results.zip